# Frequency Law v9.0 — Computational Companion

**STATUS: FROZEN**

**Christian Berrang** · [github.com/Christianfwb/Pauli-Solution-Frequency-Law](https://github.com/Christianfwb/Pauli-Solution-Frequency-Law)

> *The equations stay the same. The direction of reading changes.*

---

## Scope of this notebook

**The Acts contain the source. This notebook is the compiler.**
**It tests whether the stated relations execute consistently; it does not**
**reproduce every derivation that produced them.**

Its role is analogous to a build check or test suite: it verifies that the
equations, units, numerical transformations and stated outputs hold together
without contradiction.

The conceptual derivations and physical arguments live in Acts I–IV.
A derivation absent from this notebook is therefore not necessarily absent
from the framework — it is outside the scope of this computational layer.

All numerical identities used here are drawn from standard relations, unless
explicitly marked as **model assumption**, **geometric interpretation**, or
**candidate window**. Those marks are load-bearing: where they appear, the
framework is proposing something rather than computing something.

Run all cells. Requires `numpy` and `pandas`.

## 1 · Constants and Conventions

**Units.** Phase difference ΔΦ is measured in **cycles** (complete turns), not radians.

```
DPhi = 1   one full turn    (= 2*pi rad)
DPhi = 2   two full turns   (= 4*pi rad)   <- spinor return
```

With this convention `T = DPhi / f`, and a single turn gives `T = 1/f`,
the ordinary period. In radian notation the equivalent form is `T = dphi / (2*pi*f)`.

In [ ]:
import numpy as np
import pandas as pd

# Exact SI defining constants (2019 redefinition)
h  = 6.62607015e-34    # Planck constant [J s]     exact
c  = 299792458.0       # speed of light [m/s]      exact
eV = 1.602176634e-19   # electronvolt [J]          exact

hbar = h / (2 * np.pi)

print(f"h    = {h:.8e} J s   (exact)")
print(f"c    = {c:.0f} m/s        (exact)")
print(f"eV   = {eV:.8e} J     (exact)")
print(f"hbar = {hbar:.8e} J s")

## 2 · The Axiom Set

| ID | Statement | Reading | Status |
|---|---|---|---|
| A0 | `N := {DPhi = 0}` | Null reference — no phase advance, therefore no emergent time | posited |
| A1 | `f [Hz]` | Frequency is primary | posited |
| A2 | ΔΦ distinguishes states | Phase difference permits distinguishability | qualitative |
| A3 | `T = DPhi / f` | Time is emergent | posited |
| A4 | `E = h*f` | Energy is derived | standard (Planck) |
| A5 | `m = h*f / c^2` | Mass is bound frequency | standard (Planck + Einstein) |

A4 and A5 are textbook relations. What the framework changes is which side of
the equals sign is read as cause. A0–A3 are the framework's own commitments.

**A2 is deliberately qualitative.** Writing `I ∝ ΔΦ` would require a definition
of `I` with stated units; none is offered here, so the axiom is kept as a
distinguishability claim rather than a proportionality.

**A0 does not by itself exclude mass.** That would require showing that mass
necessarily involves nonzero frequency and a particular phase structure —
which A5 asserts but does not prove. The argument is in Prologue.md.

In [ ]:
def compton_frequency(mass_kg):
    """A5 read forward: f = m c^2 / h  [Hz]"""
    return mass_kg * c**2 / h

def mass_from_frequency(f_hz):
    """A5 as stated: m = h f / c^2  [kg]"""
    return h * f_hz / c**2

def energy_from_frequency(f_hz):
    """A4: E = h f  [J]"""
    return h * f_hz

def time_from_phase(delta_phi_cycles, f_hz):
    """A3: T = DPhi / f, with DPhi in cycles  [s]"""
    return delta_phi_cycles / f_hz

def MeV_to_kg(mass_MeV):
    return mass_MeV * 1e6 * eV / c**2

print("core functions defined")

### A3 in both conventions

A check that the cycle convention reproduces the ordinary period,
and that the radian form agrees.

In [ ]:
f_test = 40.0   # Hz

T_cycles  = time_from_phase(1.0, f_test)        # DPhi = 1 turn
T_radians = (2*np.pi) / (2*np.pi*f_test)        # dphi = 2 pi rad
T_period  = 1.0 / f_test

print(f"cycles  convention : {T_cycles:.6f} s")
print(f"radian  convention : {T_radians:.6f} s")
print(f"ordinary period    : {T_period:.6f} s")
print()
print("agree:", np.isclose(T_cycles, T_period) and np.isclose(T_radians, T_period))

## 3 · Consistency Check — Not Evidence

**Read this before running the next cell.**

Computing a particle's mass from its Compton frequency returns the measured value
exactly. This is **not** confirmation of anything.

The Compton frequency is *defined* through the mass. The calculation returns what was
put in. It is a necessary consistency check — the framework would be broken without it —
but it carries no evidential weight whatsoever.

This is the clearest example of what this notebook can and cannot do.
Section 8 says why.

In [ ]:
m_e = 9.1093837015e-31              # kg, PDG electron mass
f_e = compton_frequency(m_e)
m_e_back = mass_from_frequency(f_e)

print("CONSISTENCY CHECK (round trip, not a test)")
print("-" * 46)
print(f"electron mass in    : {m_e:.10e} kg")
print(f"Compton frequency   : {f_e:.10e} Hz")
print(f"electron mass out   : {m_e_back:.10e} kg")
print(f"relative difference : {abs(m_e_back - m_e)/m_e:.2e}")
print()
print("Exact by construction. This is arithmetic, not evidence.")

## 4 · Frequency Ordering of Particles

Particles ordered by Compton frequency. Three columns carry the caveats:

- `mass_source` — where the mass value comes from
- `topology` — **model assignment**, not a measured property
- `status` — how the entry should be read

**Topology assignment rule (model):** half-integer spin → two-turn return (`Mobius`);
integer spin → one-turn return (`circle`). This restates the spin–statistics
distinction in the framework's vocabulary. It is not an independent result.

**On quark masses:** quarks are not observed as free particles, and their masses
are scheme- and scale-dependent. They are included for completeness but are not
the same kind of quantity as the electron or proton mass.

In [ ]:
# name, mass MeV/c^2, spin, topology (model), mass_source, status
PARTICLES = [
    ("Neutrino benchmark", 2e-9,     0.5, "Mobius", "assumed benchmark",  "ASSUMPTION"),
    ("Electron",           0.511,    0.5, "Mobius", "PDG, direct",        "measured"),
    ("Berrangium Omega",   16.2,     0.5, "Mobius", "framework candidate","CANDIDATE"),
    ("Muon",               105.7,    0.5, "Mobius", "PDG, direct",        "measured"),
    ("Stoecker Sigma",     530.0,    0.0, "circle", "framework candidate","CANDIDATE"),
    ("Proton",             938.3,    0.5, "Mobius", "PDG, direct",        "measured"),
    ("Tau",                1777.0,   0.5, "Mobius", "PDG, direct",        "measured"),
    ("Bottom quark",       4180.0,   0.5, "Mobius", "PDG, MS-bar scheme", "scheme-dep"),
    ("W boson",            80400.0,  1.0, "circle", "PDG, direct",        "measured"),
    ("Higgs",              125100.0, 0.0, "circle", "PDG, direct",        "measured"),
    ("Top quark",          172700.0, 0.5, "Mobius", "PDG, pole mass",     "scheme-dep"),
]

rows = []
for name, m_MeV, spin, topo, source, status in PARTICLES:
    rows.append({
        "particle":     name,
        "mass_MeV":     f"{m_MeV:g}",
        "spin":         spin,
        "f_compton_Hz": f"{compton_frequency(MeV_to_kg(m_MeV)):.3e}",
        "topology":     topo + " (model)",
        "mass_source":  source,
        "status":       status,
    })

df = pd.DataFrame(rows)
df = df.iloc[df["f_compton_Hz"].map(float).argsort()].reset_index(drop=True)
print(df.to_string(index=False))
print()
print("Masses are rounded for display. Uncertainties are not shown.")
print("Consult the current PDG Review of Particle Physics for precise values.")

### On the neutrino entry

Neutrino oscillation experiments determine mass-squared *differences*,
not the absolute mass of the lightest state. A value of 2 meV is a
**benchmark assumption**, not a measurement.

The arithmetic below is worth checking regardless, since this entry spans
nine orders of magnitude from the electron and an earlier version of this
table got it wrong by a factor of 1000 (meV read as eV).

In [ ]:
E_nu = 2e-3 * eV                                       # 2 meV in joules
f_nu_via_energy = E_nu / h                             # A4
f_nu_via_mass   = compton_frequency(MeV_to_kg(2e-9))   # A5

print("neutrino benchmark, 2 meV")
print(f"  via E = h f     : {f_nu_via_energy:.4e} Hz")
print(f"  via f = m c^2/h : {f_nu_via_mass:.4e} Hz")
print(f"  agree           : {np.isclose(f_nu_via_energy, f_nu_via_mass)}")
print()
print("Correct order of magnitude: 1e11 Hz, not 1e8.")

## 5 · Where the Factor of 2 Appears

This section was **wrong in earlier drafts** and is corrected here.

### What spinor periodicity does establish

A spin-1/2 state changes sign under a spatial rotation of one full turn,
and returns to its original spinor only after two:

```
psi(2*pi) = -psi(0)
psi(4*pi) = +psi(0)
```

This is the 4π periodicity of SU(2). Correct and standard.

### What it does *not* establish

It does **not** derive the Zitterbewegung frequency.

An earlier version argued that |ψ|² returns after one turn while ψ needs two,
giving a factor of 2. That argument is empty: a global sign disappears from
|ψ|² entirely, so |ψ|² is constant at every angle. It never left, so it cannot
"return twice as often." The next cell shows this explicitly.

### Where the factor actually comes from

In the standard Dirac account, Zitterbewegung arises from interference between
positive- and negative-energy components. At rest their separation is:

```
Delta_E  = (+m*c^2) - (-m*c^2) = 2*m*c^2
omega_ZB = Delta_E / hbar      = 2*m*c^2 / hbar
f_ZB     = omega_ZB / (2*pi)   = 2*m*c^2 / h  = 2 * f_Compton
```

### The framework's actual claim

The Frequency Law proposes a **geometric interpretation** of this same factor.
That interpretation is a hypothesis, argued in Act III. The numerical factor
itself follows from the Dirac energy separation and would hold with or without
the geometric reading.

Whether a 4π *rotation* period and a 2mc² *energy* separation are two faces of
one structure — or merely two places where the number 2 appears — is exactly
the open question. This notebook cannot settle it; it can only confirm that
both numbers are what they are claimed to be.

In [ ]:
def spinor_phase(delta_phi_cycles):
    """SU(2): a spin-1/2 state accumulates half the turn angle.
    One turn -> exp(i pi) = -1.  Two turns -> +1."""
    return np.exp(1j * np.pi * delta_phi_cycles)

print("Spinor periodicity — and why |psi|^2 proves nothing here")
print("-" * 56)
print(f"{'turns':>6} {'psi/psi0':>18} {'|psi|^2':>10}")
for turns in [0.0, 0.5, 1.0, 1.5, 2.0]:
    a = spinor_phase(turns)
    print(f"{turns:>6.1f} {a.real:>9.3f}{a.imag:>+8.3f}i {abs(a)**2:>10.3f}")

print()
print("|psi|^2 is 1.000 at every angle. A global phase is unobservable.")
print("No factor of 2 can be extracted from this column.")

In [ ]:
# The factor of 2, from the Dirac energy separation
E_plus  =  m_e * c**2
E_minus = -m_e * c**2
delta_E = E_plus - E_minus          # = 2 m c^2

omega_zb = delta_E / hbar
f_zb     = omega_zb / (2 * np.pi)

print("Zitterbewegung from positive/negative energy interference")
print("-" * 56)
print(f"Delta_E                  : {delta_E:.6e} J   (= 2 m c^2)")
print(f"omega_ZB                 : {omega_zb:.6e} rad/s")
print()
print(f"Compton frequency        : {f_e:.6e} Hz")
print(f"Zitterbewegung frequency : {f_zb:.6e} Hz")
print(f"ratio                    : {f_zb / f_e:.6f}")
print()
print("Quantum-simulated with a trapped ion:")
print("Gerritsma et al., Nature 463 (2010).")
print("Note: this simulated Dirac dynamics; it is not a direct measurement")
print("of Zitterbewegung in a free electron.")

## 6 · Pauli Exclusion

Quantum mechanics requires the two-fermion state to be antisymmetric under
exchange. Written properly, for one-particle states `a`, `b` at coordinates
`x1`, `x2`:

```
Psi(x1, x2) = [ psi_a(x1) psi_b(x2) - psi_a(x2) psi_b(x1) ] / sqrt(2)
```

If both fermions occupy the same one-particle state (`a = b`), the two terms
are identical and cancel exactly:

```
Psi(x1, x2) = 0
```

That is Pauli exclusion — zero by construction, not by decree.

### What the framework adds, and what it does not

The Frequency Law proposes reading fermionic exchange geometrically, as a half
traversal of a non-orientable surface. That reading is developed in Act IV.

Standard quantum mechanics *requires* antisymmetry under exchange and derives
exclusion from it. Whether the Möbius description **explains** that antisymmetry,
rather than **restating** it in different vocabulary, is a separate claim —
and this notebook does not establish it either way.

In [ ]:
def antisymmetrized(psi_a_x1, psi_a_x2, psi_b_x1, psi_b_x2):
    """Antisymmetrized two-fermion amplitude (2x2 Slater determinant)."""
    return (psi_a_x1 * psi_b_x2 - psi_a_x2 * psi_b_x1) / np.sqrt(2)

# Case 1: both fermions in the SAME one-particle state
psi_x1, psi_x2 = 1.0 + 0.0j, 0.5 + 0.5j
same = antisymmetrized(psi_x1, psi_x2, psi_x1, psi_x2)

# Case 2: two DISTINCT one-particle states
distinct = antisymmetrized(1.0 + 0.0j, 0.5 + 0.5j,
                           0.2 + 0.1j, 0.8 - 0.1j)

print("same one-particle state for both fermions")
print(f"  Psi     = {same}")
print(f"  |Psi|^2 = {abs(same)**2:.6f}   -> excluded")
print()
print("distinct one-particle states")
print(f"  Psi     = {distinct}")
print(f"  |Psi|^2 = {abs(distinct)**2:.6f}   -> allowed")
print()
print("Case 1 vanishes for ANY choice of psi. That is the content of the principle.")

## 7 · Candidate Mass Windows

**These are not independent predictions in their present form.**

Both values sit near known experimental anomalies, and were identified with
those anomalies in view. Until a derivation exists that produces these numbers
*without* reference to the anomaly data, they are retrodictive candidates —
assignments of existing puzzles to gaps in the frequency ordering.

That is a weaker claim than "prediction," and it is the honest one.

### What would turn these into predictions

1. A derivation producing 16.2 and 530 MeV from the axioms alone
2. A stated uncertainty band, fixed in advance
3. A distinguishing signature — width, spin/parity, decay channels, couplings
4. An explicit exclusion criterion: what result would rule the candidate out

Points 1 and 3 are the open work.

In [ ]:
candidates = [
    ("Berrangium Omega", 16.2,
     "between electron (0.511 MeV) and muon (105.7 MeV)",
     "X17 anomaly, Atomki Institute, near 17 MeV; replicated, unexplained",
     "no derivation yet independent of the anomaly"),
    ("Stoecker Sigma", 530.0,
     "between muon (105.7 MeV) and proton (938.3 MeV)",
     "f0(500) resonance: known, very broad, poor fit to the quark model",
     "f0(500) already exists — a new state would need a distinguishing signature"),
]

for name, m_MeV, position, context, caveat in candidates:
    f = compton_frequency(MeV_to_kg(m_MeV))
    print(name)
    print(f"  candidate mass    : {m_MeV} MeV/c^2")
    print(f"  Compton frequency : {f:.4e} Hz")
    print(f"  position          : {position}")
    print(f"  existing anomaly  : {context}")
    print(f"  open issue        : {caveat}")
    print()

## 8 · What the Equations Can and Cannot Show

### Verified in this notebook

- A3 in cycle convention reproduces the ordinary period exactly
- The mass/frequency round trip is consistent — and evidentially empty
- Spinors have 4π periodicity (standard SU(2))
- Zitterbewegung is 2·f_Compton, from the Dirac energy separation ΔE = 2mc²
- Pauli exclusion follows from antisymmetry of the two-fermion state
- Every implemented relation and numerical check in this notebook executes consistently

### Outside the scope of this notebook

- **Whether the 4π rotation period and the 2mc² energy separation are the same 2.**
  Both contain a factor of two. Act III argues they share a cause; this file
  only confirms both numbers are what they are claimed to be.
- **Whether the Möbius reading explains antisymmetry** or restates it. See Act IV.
- **Whether frequency is causally prior to mass.** See Prologue.md.
- **Whether the two candidate masses correspond to real particles.** See experiment.

---

### Why calculation cannot settle those questions

Every cell above produced the right number. None of them tells you why.

This is not a defect of the notebook. It is a property of what equations are.

A compiled program runs. Watching it pass a test tells you that the tested path
worked. It does not hand you the source. The variable names are gone, the intent
is gone, the structure that produced the behaviour has been flattened into
behaviour. You can read the output forever without recovering what generated it.

Physical equations are in the same position. `m = hf/c²` is the compiled artifact.
It is exact, it is verifiable, and it is silent on the one question this framework
asks — which side is the cause.

That silence is symmetric. It does not favour either causal reading.
The relation `f = mc²/h` follows algebraically from `E = hf` and `E = mc²`;
what is inherited rather than derived is the interpretation that mass is primary
and frequency merely descriptive. The equation permits both algebraic readings
and, by itself, endorses neither causal direction.

**Numerical agreement therefore cannot decide the question. Nothing in sections
1 through 6 could have, no matter how precise the arithmetic.** That is why the
consistency check in section 3 is marked as carrying no weight.

Only a case where the two readings predict *different* things can settle it.
Section 7 is where such a case would have to be built.

---

### The honest position

This notebook does not itself reproduce a derivation that standard physics lacks.
Its purpose is narrower: to test whether the computational consequences stated in
Acts I–IV are internally consistent.

The implemented checks are internally consistent. Whether the source establishes
more than a consistent reinterpretation must be judged in the Acts themselves.
This file only verifies that the compiled relations run without contradiction.

The argument for why the question is worth asking is in [Prologue.md](Prologue.md).

---

> *The equation is symmetric. The claim about causality is not.*

> *Watching the program pass its tests shows that the tested paths are consistent.*
> *It does not give you back the source.*